# 🛠️ Phase 3: Environment Setup

Welcome to the **LLM-Studio Training Module**! This notebook handles environment verification and dependency installation. It supports execution on both local setups and **Google Colab** environments with GPU acceleration.

## Requirements Overview
We use parameter-efficient fine-tuning (PEFT) with **QLoRA** (4-bit quantized Low-Rank Adaptation). This requires the following libraries:
- `transformers` for base models loading and prompt tokenization
- `peft` for adapter scaling overlays
- `trl` for supervised fine-tuning (`SFTTrainer`)
- `bitsandbytes` for double 4-bit float quantizations
- `accelerate` for multi-GPU memory orchestration
- `datasets` for caching and loading splits
- Visualizations and score metrics helper dependencies (`matplotlib`, `seaborn`, `rouge-score`)

## 1. Install Dependencies
Run the following cell to install the latest mutually compatible library versions. If you are running on Google Colab, you will need to restart the session after installation.

In [1]:
# Check if running on Google Colab or locally
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running on Google Colab. Installing dependencies...")
    !pip install -q transformers>=4.45.0 trl>=0.11.0 peft>=0.12.0 datasets>=3.0.0 accelerate>=0.34.0 bitsandbytes>=0.43.0 rouge-score pandas matplotlib seaborn scikit-learn
    print("Dependencies installed! Please restart runtime if prompted.")
else:
    print("To install dependencies locally, run: pip install -r training/requirements.txt")

Running on Google Colab. Installing dependencies...
Dependencies installed! Please restart runtime if prompted.


### Environment Summary
Let's display the versions of Python, PyTorch, and Hugging Face deep learning libraries installed in this session.

In [2]:
import sys
import torch

print(f"Python version: {sys.version.split()[0]}")
print(f"Torch version: {torch.__version__}")

for lib in ["transformers", "peft", "trl", "datasets", "accelerate", "bitsandbytes"]:
    try:
        mod = __import__(lib)
        print(f"{lib.capitalize()} version: {mod.__version__}")
    except ImportError:
        print(f"{lib.capitalize()} version: Not installed")

print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Total GPU memory: {props.total_memory / (1024 ** 3):.2f} GB")
else:
    print("GPU name: N/A (CPU Mode)")
    print("Total GPU memory: N/A")

Python version: 3.12.13
Torch version: 2.11.0+cu128
Transformers version: 5.13.1
Peft version: 0.19.1
Trl version: 1.9.0
Datasets version: 5.0.0
Accelerate version: 1.14.0
Bitsandbytes version: 0.49.2
CUDA version: 12.8
GPU name: Tesla T4
Total GPU memory: 14.56 GB


## 2. Check Hardware Acceleration
QLoRA requires a CUDA-compatible GPU (T4, V100, A10G, or A100 are common in Google Colab). The cell below checks your CUDA availability, driver spec, and VRAM memory sizes.

In [3]:
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    if props.major >= 8:
        print("GPU supports bfloat16 precision. You can configure 'training.bf16: true' in qlora_config.yaml.")
    else:
        print("GPU does not native support bfloat16 precision. Standard fp16 training is recommended.")
    !nvidia-smi
else:
    print("WARNING: No GPU found. Training model weights on CPU is highly discouraged due to compute constraints.")

CUDA Available: True
GPU does not native support bfloat16 precision. Standard fp16 training is recommended.
Thu Jul 23 10:14:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                               

## 3. Mount Google Drive and Verify Workspace
We mount Google Drive to `/content/drive` and verify/create only the required workspace folders inside our persistent directory structure.

In [4]:
from pathlib import Path
import subprocess

if IN_COLAB:
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/ChetanVK10/LLM-Studio.git"
    PROJECT_ROOT = Path("/content/LLM-Studio")
    RUNTIME_ROOT = Path("/content/drive/MyDrive/LLM-Studio")

    if not PROJECT_ROOT.exists():
        print("Cloning latest project from GitHub...")
        subprocess.run(
            ["git", "clone", REPO_URL, str(PROJECT_ROOT)],
            check=True
        )
    else:
        print("Updating project from GitHub...")
        subprocess.run(
            ["git", "-C", str(PROJECT_ROOT), "fetch", "origin"],
            check=True
        )
        subprocess.run(
            ["git", "-C", str(PROJECT_ROOT), "reset", "--hard", "origin/main"],
            check=True
        )
        subprocess.run(
            ["git", "-C", str(PROJECT_ROOT), "clean", "-fd"],
            check=True
        )

else:
    print("Running locally...")
    PROJECT_ROOT = Path("../../").resolve()
    RUNTIME_ROOT = PROJECT_ROOT

# ------------------------------------------------------------------
# Verify source code directories (PROJECT_ROOT)
# ------------------------------------------------------------------

required_project_dirs = [
    "training",
    "training/configs",
    "training/notebooks",
    "training/scripts",
]

print("\n========== SOURCE CODE VERIFICATION ==========\n")
print(f"PROJECT_ROOT (GitHub Source) : {PROJECT_ROOT.resolve()}\n")

missing = []
for folder in required_project_dirs:
    path = PROJECT_ROOT / folder
    if path.exists():
        print(f"✓ {folder}")
    else:
        print(f"✗ {folder}")
        missing.append(folder)

if missing:
    raise FileNotFoundError(
        "\nThe following required source folders are missing:\n"
        + "\n".join(missing)
    )

print("\n✓ Source structure verified successfully.\n")

# ------------------------------------------------------------------
# Prepare persistent runtime directories (RUNTIME_ROOT)
# ------------------------------------------------------------------

runtime_dirs = [
    "data/raw",
    "data/processed",
    "models/adapters",
    "models/merged",
    "artifacts/checkpoints",
    "artifacts/evaluations",
    "artifacts/experiments",
    "logs",
]

print("========== RUNTIME WORKSPACE VERIFICATION ==========\n")
print(f"RUNTIME_ROOT (Google Drive)  : {RUNTIME_ROOT.resolve()}\n")
for folder in runtime_dirs:
    dir_path = RUNTIME_ROOT / folder
    dir_path.mkdir(parents=True, exist_ok=True)
    print(f"✓ {folder}")

# Common path pointers
CONFIG_DIR = PROJECT_ROOT / "training/configs"
SCRIPT_DIR = PROJECT_ROOT / "training/scripts"
DATA_DIR = RUNTIME_ROOT / "data"
MODELS_DIR = RUNTIME_ROOT / "models"
ARTIFACTS_DIR = RUNTIME_ROOT / "artifacts"

print("\n========== PATH SUMMARY ==========")
print(f"PROJECT_ROOT : {PROJECT_ROOT.resolve()}")
print(f"RUNTIME_ROOT : {RUNTIME_ROOT.resolve()}")
print(f"Configs      : {CONFIG_DIR}")
print(f"Scripts      : {SCRIPT_DIR}")
print(f"Data         : {DATA_DIR}")
print(f"Models       : {MODELS_DIR}")
print(f"Artifacts    : {ARTIFACTS_DIR}")

Mounted at /content/drive
Cloning latest project from GitHub...

========== SOURCE CODE VERIFICATION ==========

PROJECT_ROOT (GitHub Source) : /content/LLM-Studio

✓ training
✓ training/configs
✓ training/notebooks
✓ training/scripts

✓ Source structure verified successfully.

========== RUNTIME WORKSPACE VERIFICATION ==========

RUNTIME_ROOT (Google Drive)  : /content/drive/MyDrive/LLM-Studio

✓ data/raw
✓ data/processed
✓ models/adapters
✓ models/merged
✓ artifacts/checkpoints
✓ artifacts/evaluations
✓ artifacts/experiments
✓ logs

========== PATH SUMMARY ==========
PROJECT_ROOT : /content/LLM-Studio
RUNTIME_ROOT : /content/drive/MyDrive/LLM-Studio
Configs      : /content/LLM-Studio/training/configs
Scripts      : /content/LLM-Studio/training/scripts
Data         : /content/drive/MyDrive/LLM-Studio/data
Models       : /content/drive/MyDrive/LLM-Studio/models
Artifacts    : /content/drive/MyDrive/LLM-Studio/artifacts


## 4. Hugging Face Authentication
To securely authenticate and download base model weights, we use standard Hugging Face APIs.

In [5]:
from huggingface_hub import login, whoami

token = None
if IN_COLAB:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        if token:
            print("Using HF_TOKEN secret found in Google Colab Secrets (userdata)...")
    except Exception:
        pass

if token:
    login(token=token)
else:
    print("Hugging Face token not found in secrets. Launching interactive login widget...")
    login()

# Verify login succeeded
try:
    user_info = whoami()
    print(f"\n✓ Hugging Face authentication succeeded! Logged in as: {user_info['name']}")
except Exception as e:
    print(f"\n❌ Hugging Face login failed: {e}")

Hugging Face token not found in secrets. Launching interactive login widget...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")



✓ Hugging Face authentication succeeded! Logged in as: ChetanVk10


## 5. Set Random Seeds
Set random seeds across Python, NumPy, and PyTorch for reproducibility.

In [6]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"✓ Configured random seed: {SEED}")

✓ Configured random seed: 42


## 6. Load Quantized Model
We load the base **Qwen/Qwen2.5-3B-Instruct** model in 4-bit precision config using bitsandbytes NF4 double quantization.

In [7]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading {model_id} model in 4-bit configuration...")
device_map = "auto" if torch.cuda.is_available() else None

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map=device_map,
    trust_remote_code=True
)

print(f"\nModel dtype: {model.dtype}")

if hasattr(model, "hf_device_map"):
    print(f"Device map: {model.hf_device_map}")
else:
    print(f"Model device: {next(model.parameters()).device}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameter count: {total_params:,}")

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / (1024**3)
    reserved = torch.cuda.memory_reserved() / (1024**3)
    print(f"GPU allocated: {allocated:.2f} GB")
    print(f"GPU reserved : {reserved:.2f} GB")
else:
    print("GPU memory usage: N/A (CPU Mode)")

Loading Qwen/Qwen2.5-3B-Instruct model in 4-bit configuration...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model dtype: torch.bfloat16
Model device: cuda:0
Parameter count: 1,698,672,640
GPU allocated: 1.92 GB
GPU reserved : 1.98 GB


## 7. Load Tokenizer
We load the tokenizer and print special token configurations and padding attributes.

In [8]:
from transformers import AutoTokenizer

print(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"Configured PAD token as EOS token: {tokenizer.pad_token}")

print(f"\nVocabulary size: {tokenizer.vocab_size}")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
max_seq_len = getattr(tokenizer, "model_max_length", None)
print(f"Maximum sequence length: {max_seq_len}")

Loading tokenizer for Qwen/Qwen2.5-3B-Instruct...


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]


Vocabulary size: 151643
EOS token: <|im_end|> (ID: 151645)
PAD token: <|endoftext|> (ID: 151643)
Maximum sequence length: 131072


## 8. Run Sanity Test Inference
Let's execute a single prediction inference generation cycle on QLoRA theory questions to ensure model loading is successful.

In [9]:
prompt = "Explain what QLoRA is in two sentences."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

print("Generating response...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

input_len = inputs.input_ids.shape[1]
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
print(f"\nGenerated response:\n{response}")

Generating response...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Generated response:
QLoRA (Quantized LoRA) is a method that quantizes the parameters of a large model to reduce memory usage and computational cost while still allowing for lightweight finetuning, enabling faster inference and more efficient training on resource-constrained devices.


# 🏁 Final Checklist Completed

✓ Google Drive mounted

✓ Workspace verified

✓ Dependencies installed

✓ GPU detected

✓ Hugging Face authenticated

✓ Tokenizer loaded

✓ Model loaded

✓ Test inference successful

✓ Notebook 1 completed successfully